# 8 · Loading data — every input roadstyle accepts

`render_edges` / `to_spec` don't only take a GeoDataFrame. Everything funnels through `as_edges`, which normalises **any** of the sources below to the one canonical shape (`RoadEdges`: EPSG:4326 line geometry + a class column). So you can hand roadstyle whatever you already have.

In [1]:
import pathlib
import geopandas as gpd
import roadstyle as rs

# the sample we'll re-load from every source below
edges = gpd.read_file(pathlib.Path("data") / "sodermalm_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")


4,563 edges, CRS EPSG:4326


ERROR 1: PROJ: proj_create_from_database: Open of /home/kaveh/miniconda3/envs/roadstyle/share/proj failed


### A GeoDataFrame
The baseline — what the other examples use.

In [2]:
rs.render_edges(edges, basemap="dark_matter", compress=True)          # the baseline: a GeoDataFrame

### A file path
Give it a path and it loads the file (`load_edges` under the hood).

In [3]:
# A file path — GeoPackage / GeoJSON / Shapefile / … is loaded for you.
rs.render_edges(pathlib.Path("data") / "sodermalm_edges.gpkg", basemap="dark_matter", compress=True)


### GeoJSON — a FeatureCollection, or a roadstyle spec
A GeoJSON mapping (FeatureCollection / Feature / geometry / `__geo_interface__`) — and a `to_spec()` dict round-trips back in, so a saved spec can be re-rendered.

In [4]:
import json

# Any GeoJSON FeatureCollection / Feature / geometry mapping works:
fc = json.loads(edges.head(80).to_json())
rs.render_edges(fc, basemap="dark_matter", compress=True)

# A roadstyle to_spec() dict round-trips straight back in:
spec = rs.to_spec(edges)
rs.render_edges(spec, compress=True)


### A pyarrow Table
GeoArrow tables work directly; for a plain WKB/WKT column, name it with `from_arrow(table, geometry=…, crs=…)`.

In [5]:
import pyarrow as pa

# A GeoArrow table carries its CRS, so it works directly:
geoarrow = pa.table(edges.head(80).to_arrow())
rs.render_edges(geoarrow, basemap="dark_matter", compress=True)

# A plain table with a WKB (or WKT) geometry column: name the column + its CRS.
table = pa.table({
    "highway": edges["highway"].astype(str).tolist(),
    "geometry": [g.wkb for g in edges.geometry],
})
rs.render_edges(rs.from_arrow(table, geometry="geometry", crs=4326), basemap="dark_matter", compress=True)


### DuckDB — a relation, or a connection + query
Select the geometry as **WKB** (`ST_AsWKB`) or **WKT** (`ST_AsText`) so it round-trips. Pass `crs=` because DuckDB doesn't carry one. This is the path for pulling straight from a DuckDB analytics table.

In [6]:
# Needs DuckDB:  pip install duckdb   (or the extra:  pip install "roadstyle[duckdb]")
import duckdb

con = duckdb.connect()
con.execute("CREATE TABLE roads (highway VARCHAR, geom BLOB)")   # store geometry as WKB
con.executemany(
    "INSERT INTO roads VALUES (?, ?)",
    [(str(h), g.wkb) for h, g in zip(edges["highway"], edges.geometry)],
)

# Pass a connection + query (select geometry as WKB / WKT), or a relation (con.sql(...)).
# DuckDB carries no CRS, so `crs=` says what the coordinates are in.
e = rs.from_duckdb(con, "SELECT highway, geom FROM roads", geometry="geom", crs=4326)
rs.render_edges(e, color_by="highway", basemap="dark_matter", compress=True)


### In short
| You have… | Just pass it | For options use |
|---|---|---|
| GeoDataFrame / `RoadEdges` | `render_edges(gdf)` | `normalize_edges` |
| a file | `render_edges("roads.gpkg")` | `load_edges` |
| GeoJSON / spec | `render_edges(fc)` | `from_geojson` |
| pyarrow Table | `render_edges(table)` | `from_arrow` |
| DuckDB | — | `from_duckdb` |

All return / accept the canonical `RoadEdges`; reach for the `from_*` helpers when you need to name the geometry column or set the CRS.